# Phase 3 — Extensions across arms (Gray-Scott)

Loads every arm that's available and runs all extensions on each, with
comparison plots:
- **CNN** ("LiteFNO" class in the repo) — from the committed checkpoint (always)
- **Real LiteFNO** — from Phase 2's `litefno_real_best.pt` (if mounted)
- **FNO-S** — optional; only if you point `FNOS_CKPT` at one

Checkpoint-based extensions: sanity, quantization, noise robustness,
autoregressive rollout (+windowed VRMSE), input spectral sensitivity, energy
spectrum, error maps, inference benchmark. Plus the **logs-only 8-dataset
analysis** (no checkpoint needed).

Run it now for the CNN arm (parallel with Phase 2); re-run after Phase 2 with the
real-LiteFNO checkpoint mounted to fill in that arm.

**Setup:** Internet ON; GPU optional.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/AIscend-Research/litefno-repro"
REPO_DIR = "litefno-repro"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "neuraloperator", "the_well", "thop"], check=False)
print("cwd:", os.getcwd())

In [ ]:
import json, time, copy, csv
from pathlib import Path
import numpy as np
import torch, h5py
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":   # some Kaggle GPUs (P100, sm_60) are incompatible with new torch
    try:
        _ = (torch.zeros(1, device=DEVICE) + 1).item()
    except Exception as e:
        print("CUDA unusable, falling back to CPU:", e); DEVICE = torch.device("cpu")
OUT = Path("/kaggle/working/extensions") if Path("/kaggle/working").exists() else Path("extensions_out")
OUT.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE, "| out:", OUT.resolve())

DATASET="gray_scott_reaction_diffusion"; KEY="data"; FIELDS=2
DOWNSAMPLE=4; MAX_TRAJ=1000; MAX_STEPS=60; SEED=0
RAW=Path("/kaggle/temp/gs_raw") if Path("/kaggle").exists() else Path("data/raw/gs_ext"); PROC=Path("data/processed/gs_ext"); TEST_H5=PROC/"test.h5"

CNN_CKPT  = Path("outputs/checkpoints/gray_scott_reaction_diffusion/litefno/best.pt")
# Point these at mounted Kaggle inputs after Phase 2 (edit as needed):
REAL_CKPT = Path("/kaggle/input/phase2-output/litefno_real_best.pt")
FNOS_CKPT = None   # optional: set to a checkpoint path string to include FNO-S in figures

## Download + preprocess GS test split

In [ ]:
from litefno.download import download_dataset
from litefno.preprocess import preprocess_well_split
import glob as _g
# Prefer a mounted preprocessed test.h5 (matches Phase 2's training distribution -> fair eval).
_test_hits = sorted(_g.glob("/kaggle/input/**/test.h5", recursive=True))
DATA_OK = TEST_H5.exists()
if _test_hits and not DATA_OK:
    TEST_H5 = Path(_test_hits[0])
    print("Using mounted test set (matches Phase 2 training regimes):", TEST_H5)
    DATA_OK = True
if not DATA_OK:
    try:
        RAW.mkdir(parents=True, exist_ok=True)
        download_dataset(DATASET, "test", RAW)
        PROC.mkdir(parents=True, exist_ok=True)
        preprocess_well_split(RAW, TEST_H5, DATASET, "test", KEY, DOWNSAMPLE, MAX_TRAJ, MAX_STEPS, random_seed=SEED)
        import shutil
        raw_split = RAW / "datasets" / DATASET / "data" / "test"
        if raw_split.exists(): shutil.rmtree(raw_split)
        DATA_OK = True
    except Exception as e:
        print("!! GS test download/preprocess failed:", repr(e)); DATA_OK = False
if DATA_OK:
    with h5py.File(TEST_H5, "r") as f:
        TEST = f[KEY][...].astype(np.float32)
    H, W = TEST.shape[2], TEST.shape[3]
    print("test:", TEST.shape)

## Load every available arm

All arms share the same `(B,C,H,W) -> (B,C,H,W)` interface, so the extension code is identical across them.

In [ ]:
def build_real_litefno(in_ch, out_ch, modes, width=64, layers=8, rank=0.5, factorization="cp"):
    """Real spectral LiteFNO: a CP/Tucker-factorized FNO (neuraloperator).

    Tries the requested factorization, then tucker, then a dense FNO, so the
    notebook still produces a genuine *spectral* operator even if a particular
    factorization API is unavailable. Returns (model, kind_used).
    """
    from neuralop.models import FNO
    base = dict(n_modes=(modes, modes), hidden_channels=width,
                in_channels=in_ch, out_channels=out_ch, n_layers=layers)
    fac = None if factorization in (None, "dense") else factorization
    attempts = [(fac, rank), ("tucker", rank), (None, None)]
    last = None
    for f, r in attempts:
        try:
            if f is None:
                return FNO(**base), "dense"
            return FNO(**base, factorization=f, rank=r), f
        except Exception as e:  # noqa: BLE001
            last = e
            print(f"  [build] factorization={f} failed: {e}")
    raise RuntimeError(f"could not construct FNO: {last}")

from litefno.train import build_model, load_checkpoint, count_parameters

ARMS = {}      # name -> model
PARAMS = {}    # name -> param count

if DATA_OK:
    # CNN arm (repo)
    if CNN_CKPT.exists():
        m = build_model({"name": "litefno", "layers": 8, "width": 64, "rank": 32}, FIELDS, FIELDS)
        load_checkpoint(CNN_CKPT, m, device=DEVICE); m.to(DEVICE).eval()
        ARMS["cnn"] = m; PARAMS["cnn"] = count_parameters(m)

    # Real spectral LiteFNO arm (Phase 2)
    if REAL_CKPT.exists():
        ck = torch.load(REAL_CKPT, map_location=DEVICE, weights_only=False); b = ck["build"]
        m, _ = build_real_litefno(b["in_ch"], b["out_ch"], b["modes"], b["width"], b["layers"], b["rank"], b["factorization"])
        m.load_state_dict(ck["model_state"]); m.to(DEVICE).eval()
        ARMS["litefno_real"] = m; PARAMS["litefno_real"] = sum(p.numel() for p in m.parameters())

    # Optional FNO-S arm
    if FNOS_CKPT and Path(FNOS_CKPT).exists():
        m = build_model({"name": "fno_s", "layers": 8, "width": 64, "modes": 12}, FIELDS, FIELDS)
        load_checkpoint(FNOS_CKPT, m, device=DEVICE); m.to(DEVICE).eval()
        ARMS["fno_s"] = m; PARAMS["fno_s"] = count_parameters(m)

print("arms:", {k: f"{v:,}" for k, v in PARAMS.items()} or "NONE (need data + checkpoints)")

## Shared eval helpers

In [ ]:
@torch.no_grad()
def step_predict(m, state):           # state (B,H,W,C) -> (B,H,W,C)
    return m(state.permute(0, 3, 1, 2)).permute(0, 2, 3, 1)

@torch.no_grad()
def eval_one_step(m, data, bs=256, transform=None):
    N, S = data.shape[0], data.shape[1]; rs = []; vs = []
    for t in range(S - 1):
        x = data[:, t]; y = data[:, t + 1]
        if transform is not None: x = transform(x)
        xt = torch.from_numpy(np.ascontiguousarray(x)).float()
        yt = torch.from_numpy(np.ascontiguousarray(y)).float()
        for i in range(0, N, bs):
            p = step_predict(m, xt[i:i+bs].to(DEVICE)); yb = yt[i:i+bs].to(DEVICE)
            rs.append(rmse(p, yb).item()); vs.append(vrmse(p, yb).item())
    return float(np.mean(rs)), float(np.mean(vs))

from litefno.metrics import rmse, vrmse, window_vrmse
def savecsv(name, rows):
    if not rows: return
    with open(OUT / name, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)

## Extension 1 — Sanity check (per arm)

In [ ]:
if ARMS:
    rows = []
    for name, m in ARMS.items():
        r, v = eval_one_step(m, TEST)
        rows.append({"arm": name, "params": PARAMS[name], "test_rmse": r, "test_vrmse": v})
        print(f"{name:>13}: RMSE={r:.5f}  VRMSE={v:.5f}  params={PARAMS[name]:,}")
    savecsv("ext1_sanity.csv", rows)

## Extension 2 — Precision / quantization sweep (per arm)

In [ ]:
def q_real(w, bits):
    if bits == 16: return w.half().float()
    qmax = 2 ** (bits - 1) - 1; s = w.abs().max() / qmax
    return w if s == 0 else torch.clamp(torch.round(w / s), -qmax - 1, qmax) * s

def quantize(m, bits):
    mq = copy.deepcopy(m)
    with torch.no_grad():
        for p in mq.parameters():
            if bits >= 32: continue
            if p.is_complex():
                p.data = torch.complex(q_real(p.data.real, bits), q_real(p.data.imag, bits))
            else:
                p.data = q_real(p.data, bits)
    return mq

if ARMS:
    rows = []; fig, ax = plt.subplots(figsize=(6, 4))
    for name, m in ARMS.items():
        ser = []
        for bits in [32, 16, 8, 6, 4, 3, 2]:
            _, v = eval_one_step(quantize(m, bits), TEST)
            rows.append({"arm": name, "bits": bits, "vrmse": v,
                         "size_mb": PARAMS[name] * bits / 8 / 1e6})
            ser.append((bits, v))
        ax.plot([b for b, _ in ser], [v for _, v in ser], "o-", label=name)
    savecsv("ext2_quantization.csv", rows)
    ax.invert_xaxis(); ax.set_xlabel("weight bit-width"); ax.set_ylabel("test VRMSE")
    ax.set_title("Accuracy vs precision"); ax.legend()
    fig.tight_layout(); fig.savefig(OUT / "ext2_quantization.png", dpi=150); plt.close(fig)
    print("saved ext2_quantization.{csv,png}")

## Extension 3 — Gaussian-noise robustness (per arm)

In [ ]:
if ARMS:
    rng = np.random.default_rng(0)
    def noisy(snr):
        def t(x):
            std = x.std() * (10 ** (-snr / 20.0))
            return x + rng.normal(0, std, x.shape).astype(np.float32)
        return t
    rows = []; fig, ax = plt.subplots(figsize=(6, 4))
    for name, m in ARMS.items():
        ser = []
        for snr in [40, 30, 20, 10, 5, 0]:
            _, v = eval_one_step(m, TEST, transform=noisy(snr))
            rows.append({"arm": name, "snr_db": snr, "vrmse": v}); ser.append((snr, v))
        ax.plot([s for s, _ in ser], [v for _, v in ser], "o-", label=name)
    savecsv("ext3_noise.csv", rows)
    ax.invert_xaxis(); ax.set_xlabel("input SNR (dB)"); ax.set_ylabel("test VRMSE")
    ax.set_title("Robustness to input noise"); ax.legend()
    fig.tight_layout(); fig.savefig(OUT / "ext3_noise.png", dpi=150); plt.close(fig)
    print("saved ext3_noise.{csv,png}")

## Extension 4 — Autoregressive rollout + windowed VRMSE (per arm)

In [ ]:
@torch.no_grad()
def rollout(m, data, steps, bs=256):
    N, S = data.shape[0], data.shape[1]; steps = min(steps, S - 1)
    preds = torch.empty((N, steps) + tuple(data.shape[2:]), dtype=torch.float32)
    init = torch.from_numpy(data[:, 0]).float()
    for i in range(0, N, bs):
        s = init[i:i+bs].to(DEVICE)
        for k in range(steps):
            s = step_predict(m, s); preds[i:i+bs, k] = s.cpu()
    gt = torch.from_numpy(data[:, 1:steps + 1]).float()
    return preds, gt

if ARMS:
    STEPS = min(30, TEST.shape[1] - 1); rows = []; wins = []
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for name, m in ARMS.items():
        preds, gt = rollout(m, TEST, STEPS)
        per = [vrmse(preds[:, k], gt[:, k]).item() for k in range(STEPS)]
        for k, v in enumerate(per, 1): rows.append({"arm": name, "step": k, "vrmse": v})
        def win(a, b):
            a, b = min(a, STEPS), min(b, STEPS)
            return float("nan") if b <= a else window_vrmse(preds, gt, a, b, time_dim=1).item()
        wins.append({"arm": name, "vrmse_6_12": win(6, 12), "vrmse_13_30": win(13, 30)})
        ax.plot(range(1, STEPS + 1), per, "o-", ms=3, label=name)
    savecsv("ext4_rollout.csv", rows); savecsv("ext4_windows.csv", wins)
    for w in wins: print(w)
    ax.axvspan(6, 12, alpha=0.1, color="C1"); ax.axvspan(13, 30, alpha=0.1, color="C2")
    ax.set_xlabel("rollout step"); ax.set_ylabel("VRMSE"); ax.set_title("Error accumulation"); ax.legend()
    fig.tight_layout(); fig.savefig(OUT / "ext4_rollout.png", dpi=150); plt.close(fig)
    print("saved ext4_rollout.{csv,png}, ext4_windows.csv")

## Extension 5 — Input spectral sensitivity (per arm)

In [ ]:
if ARMS:
    yy, xx = np.ogrid[:H, :W]; cy, cx = H / 2.0, W / 2.0
    rr = np.sqrt(((yy - cy) / cy) ** 2 + ((xx - cx) / cx) ** 2)
    def lowpass(frac):
        mask = (rr <= frac)[None, ..., None]
        def t(x):
            F = np.fft.fftshift(np.fft.fft2(x, axes=(1, 2)), axes=(1, 2)) * mask
            return np.real(np.fft.ifft2(np.fft.ifftshift(F, axes=(1, 2)), axes=(1, 2))).astype(np.float32)
        return t
    rows = []; fig, ax = plt.subplots(figsize=(6, 4))
    for name, m in ARMS.items():
        ser = []
        for frac in [1.0, 0.75, 0.5, 0.25, 0.1, 0.05]:
            _, v = eval_one_step(m, TEST, transform=lowpass(frac))
            rows.append({"arm": name, "retained_frac": frac, "vrmse": v}); ser.append((frac, v))
        ax.plot([f for f, _ in ser], [v for _, v in ser], "o-", label=name)
    savecsv("ext5_spectral_sensitivity.csv", rows)
    ax.set_xlabel("retained input freq fraction"); ax.set_ylabel("test VRMSE")
    ax.set_title("Input frequency sensitivity"); ax.legend()
    fig.tight_layout(); fig.savefig(OUT / "ext5_spectral_sensitivity.png", dpi=150); plt.close(fig)
    print("saved ext5_spectral_sensitivity.{csv,png}")

## Extension 6 — Energy spectrum (per arm vs truth)

In [ ]:
def radial_psd(field):
    F = np.fft.fftshift(np.fft.fft2(field, axes=(1, 2)), axes=(1, 2))
    Pw = (np.abs(F) ** 2).mean(0); h, w = Pw.shape
    yy, xx = np.indices((h, w)); r = np.sqrt((yy - h / 2) ** 2 + (xx - w / 2) ** 2).astype(int)
    return np.bincount(r.ravel(), Pw.ravel()) / np.maximum(np.bincount(r.ravel()), 1)

if ARMS:
    fig, ax = plt.subplots(figsize=(6, 4))
    ps_true = radial_psd(TEST[:, 1][..., 0]); k = np.arange(1, len(ps_true))
    ax.loglog(k, ps_true[1:], "k-", lw=2, label="ground truth")
    rows = [{"k": int(i), "psd_true": ps_true[i]} for i in k]
    for name, m in ARMS.items():
        xb = torch.from_numpy(TEST[:, 0]).float().to(DEVICE)
        pb = step_predict(m, xb).cpu().numpy()
        ps = radial_psd(pb[..., 0]); ax.loglog(k, ps[1:], label=name)
        for j, i in enumerate(k): rows[j][f"psd_{name}"] = ps[i]
    savecsv("ext6_energy_spectrum.csv", rows)
    ax.set_xlabel("wavenumber k"); ax.set_ylabel("power"); ax.set_title("Energy spectrum (field 0)"); ax.legend()
    fig.tight_layout(); fig.savefig(OUT / "ext6_energy_spectrum.png", dpi=150); plt.close(fig)
    print("saved ext6_energy_spectrum.{csv,png}")

## Extension 7 — Spatial error maps (per arm)

In [ ]:
if ARMS:
    t_show = min(10, TEST.shape[1] - 2); n_arm = len(ARMS)
    fig, axes = plt.subplots(n_arm, 3, figsize=(9, 3 * n_arm), squeeze=False)
    xb = torch.from_numpy(TEST[:1, t_show]).float().to(DEVICE)
    true = TEST[0, t_show + 1, ..., 0]
    for r, (name, m) in enumerate(ARMS.items()):
        pred = step_predict(m, xb).cpu().numpy()[0, ..., 0]
        for c, (img, ttl) in enumerate([(true, "truth"), (pred, f"{name}"), (np.abs(true - pred), "|err|")]):
            ax = axes[r][c]; im = ax.imshow(img, cmap="viridis"); ax.set_xticks([]); ax.set_yticks([])
            if r == 0: ax.set_title(ttl)
            if c == 1: ax.set_ylabel(name)
            fig.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(f"Spatial error (t={t_show}->{t_show+1}, field 0)")
    fig.tight_layout(); fig.savefig(OUT / "ext7_error_maps.png", dpi=150); plt.close(fig)
    print("saved ext7_error_maps.png")

## Extension 8 — Inference benchmark (per arm, CPU + GPU)

In [ ]:
def bench_model(m, devname, bs_list=(1, 4, 16, 64), reps=20):
    dev = torch.device(devname); m = m.to(dev).eval(); res = []
    with torch.no_grad():
        for bs in bs_list:
            x = torch.randn(bs, FIELDS, H, W, device=dev)
            for _ in range(5): m(x)
            if devname == "cuda": torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(reps): m(x)
            if devname == "cuda": torch.cuda.synchronize()
            dt = (time.time() - t0) / reps
            res.append({"arm": "?", "device": devname, "batch": bs, "ms_per_batch": dt * 1e3, "samples_per_s": bs / dt})
    return res

if ARMS:
    rows = []; fig, ax = plt.subplots(figsize=(6.5, 4))
    for name, m in ARMS.items():
        devs = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
        for dn in devs:
            try:
                r = bench_model(m, dn)
            except Exception as e:
                print(f"  benchmark {name}/{dn} skipped: {e}"); continue
            for x in r: x["arm"] = name
            rows += r
            d = [x for x in r if x["device"] == dn]
            ax.plot([x["batch"] for x in d], [x["samples_per_s"] for x in d], "o-", label=f"{name}/{dn}")
        m.to(DEVICE)
    savecsv("ext8_benchmark.csv", rows)
    ax.set_xscale("log", base=2); ax.set_xlabel("batch size"); ax.set_ylabel("samples/s")
    ax.set_title("Inference throughput"); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(OUT / "ext8_benchmark.png", dpi=150); plt.close(fig)
    print("saved ext8_benchmark.{csv,png}")

## Freebie A - Reproducibility audit: released code vs. paper

A zero-compute reproducibility finding (the kind MLRC explicitly values): we
programmatically check whether the released implementations contain spectral
(FFT) operations, and contrast that with the paper's described architecture.

In [ ]:
import inspect
from litefno.models import litefno as _cnn_mod, fno_s as _fnos_mod

def has_spectral(mod):
    s = inspect.getsource(mod).lower()
    return ("fft" in s) or ("rfft" in s) or ("spectralconv" in s)

finding = {
    "paper_litefno_architecture": "spectral FFT + CP low-rank + transduction",
    "repo_litefno_class_is_spectral": has_spectral(_cnn_mod),
    "repo_fno_s_class_is_spectral": has_spectral(_fnos_mod),
}
print("Reproducibility finding")
print("  Paper LiteFNO  : spectral (FFT) + CP low-rank factorization + transduction")
print("  Repo 'LiteFNO' : spectral?", finding["repo_litefno_class_is_spectral"])
print("  Repo  FNO-S    : spectral?", finding["repo_fno_s_class_is_spectral"])
if not finding["repo_litefno_class_is_spectral"]:
    print("  => MISMATCH: the released 'LiteFNO' is a CNN (no FFT), not the spectral")
    print("     architecture the paper describes. This motivates our generalization study:")
    print("     does the spectral machinery actually earn its keep vs a plain low-rank CNN?")
with open(OUT / "freebieA_repro_audit.json", "w") as f:
    json.dump(finding, f, indent=2)
print("saved freebieA_repro_audit.json")

## Freebie B - FLOPs / compute-accuracy trade-off (per arm)

In [ ]:
try:
    from thop import profile
    HAVE_THOP = True
except Exception:
    HAVE_THOP = False
    print("thop unavailable (pip install thop) - skipping FLOPs")

if ARMS and HAVE_THOP:
    rows = []
    x = torch.randn(1, FIELDS, H, W).to(DEVICE)
    for name, m in ARMS.items():
        flops = None
        try:
            macs, _ = profile(copy.deepcopy(m).to(DEVICE), inputs=(x,), verbose=False)
            flops = 2 * macs
        except Exception as e:
            print(name, "FLOPs failed:", e)
        _, v = eval_one_step(m, TEST)
        rows.append({"arm": name, "params": PARAMS[name], "flops": flops, "test_vrmse": v})
        print(f"{name:>13}: params={PARAMS[name]:,}  flops={flops}  vrmse={v:.5f}")
    savecsv("freebieB_flops.csv", rows)
    pts = [r for r in rows if r["flops"]]
    if pts:
        fig, ax = plt.subplots(figsize=(6, 4))
        for r in pts:
            ax.scatter(r["flops"], r["test_vrmse"])
            ax.annotate(r["arm"], (r["flops"], r["test_vrmse"]), fontsize=8, xytext=(3, 3), textcoords="offset points")
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel("FLOPs / forward (1 sample)"); ax.set_ylabel("test VRMSE")
        ax.set_title("Compute-accuracy trade-off")
        fig.tight_layout(); fig.savefig(OUT / "freebieB_flops.png", dpi=150); plt.close(fig)
    print("saved freebieB_flops.{csv,png}")

## Logs-only analysis (8 datasets, no checkpoint needed)

In [ ]:
DATASETS = ["gray_scott_reaction_diffusion","euler_multi_quadrants_openBC","euler_multi_quadrants_periodicBC",
            "acoustic_scattering_discontinuous","active_matter","rayleigh_benard",
            "turbulent_radiative_layer_2D","viscoelastic_instability"]
SHORT = {"gray_scott_reaction_diffusion":"GS","euler_multi_quadrants_openBC":"EMQ-O","euler_multi_quadrants_periodicBC":"EMQ-P",
         "acoustic_scattering_discontinuous":"AS-SD","active_matter":"AM","rayleigh_benard":"RB",
         "turbulent_radiative_layer_2D":"TRL-2D","viscoelastic_instability":"VI"}
LOGDIR = Path("outputs/logs")
def read_log(ds, model):
    p = LOGDIR / f"{ds}_{model}.jsonl"
    if not p.exists(): return None
    rows = [json.loads(l) for l in open(p) if l.strip()]
    tr = [r for r in rows if "valid_vrmse" in r]; te = [r for r in rows if "test_vrmse" in r]
    if not tr: return None
    return {"params": tr[0].get("params"), "best_valid_vrmse": min(r["valid_vrmse"] for r in tr),
            "final_train_vrmse": tr[-1].get("train_vrmse"), "final_valid_vrmse": tr[-1].get("valid_vrmse"),
            "test_vrmse": te[-1]["test_vrmse"] if te else None,
            "curve": [(i, r["valid_vrmse"]) for i, r in enumerate(tr)]}
CNN = {d: read_log(d, "litefno") for d in DATASETS}
FNOS = {d: read_log(d, "fno_s") for d in DATASETS}

table = []
for d in DATASETS:
    l, f = CNN[d], FNOS[d]
    if not l: continue
    imp = (f["test_vrmse"] - l["test_vrmse"]) / f["test_vrmse"] * 100 if (f and f["test_vrmse"]) else None
    table.append({"dataset": SHORT[d], "cnn_params": l["params"], "cnn_test_vrmse": l["test_vrmse"],
                  "fnos_test_vrmse": f["test_vrmse"] if f else None, "improvement_pct": imp})
savecsv("logs_reproduction_table.csv", table)
for r in table: print(r)

In [ ]:
# improvement bar + efficiency frontier + convergence + generalization gap
imp = [r for r in table if r["improvement_pct"] is not None]
if imp:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar([r["dataset"] for r in imp], [r["improvement_pct"] for r in imp]); ax.axhline(0, color="k", lw=.6)
    ax.set_ylabel("CNN vs FNO-S VRMSE improvement (%)"); ax.set_title("Logs: CNN arm vs FNO-S")
    fig.tight_layout(); fig.savefig(OUT / "logs_improvement.png", dpi=150); plt.close(fig)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for d in DATASETS:
    l, f = CNN[d], FNOS[d]
    if l and l["test_vrmse"]:
        ax.scatter(l["params"], l["test_vrmse"], color="C0")
        ax.annotate(SHORT[d], (l["params"], l["test_vrmse"]), fontsize=8, xytext=(3, 3), textcoords="offset points")
    if f and f["test_vrmse"]: ax.scatter(f["params"], f["test_vrmse"], color="C1", marker="^")
ax.scatter([], [], color="C0", label="CNN"); ax.scatter([], [], color="C1", marker="^", label="FNO-S")
ax.set_xlabel("params"); ax.set_ylabel("test VRMSE"); ax.set_yscale("log"); ax.set_title("Efficiency frontier"); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "logs_frontier.png", dpi=150); plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4.5))
for d in DATASETS:
    l = CNN[d]
    if not l: continue
    ax.plot([i for i, _ in l["curve"]], [v for _, v in l["curve"]], lw=1, label=SHORT[d])
ax.set_xlabel("epoch"); ax.set_ylabel("valid VRMSE"); ax.set_yscale("log"); ax.set_title("CNN convergence"); ax.legend(ncol=2, fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "logs_convergence.png", dpi=150); plt.close(fig)
print("saved logs_*.png")

In [ ]:
print("Artifacts in", OUT.resolve())
for p in sorted(OUT.iterdir()): print("  ", p.name)